# 01. Data Preprocessing & Exploratory Data Analysis (EDA)

## Purpose
This is the **foundation** of our project. We load the raw insurance claims data, clean it, and perform initial analysis.

## Key Steps
1.  **Data Loading**: Import raw CSV files.
2.  **Data Cleaning**: Handle missing values (e.g., `?` placeholders) and correct data types.
3.  **EDA (Exploratory Data Analysis)**: Visualize distributions (Fraud vs Non-Fraud) to understand our baseline risk.
4.  **Export**: Save a clean version for feature engineering.


<a href="https://colab.research.google.com/github/mynameistatibond/claim-fraud-detection/blob/main/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
# ============================================================
# step_00 IMPORT CORE LIBRARIES
# ============================================================
# Why:
# - We need fast tabular ops (pandas) and numerical routines (numpy).
# - Visualization split: static (matplotlib/seaborn) for quick EDA; interactive (plotly) for drill-down and sharing.
# - Centralizing imports keeps dependencies explicit for the whole notebook/project.

# 1) Data manipulation & numerics
import pandas as pd    # DataFrames: read/write CSVs, joins, groupby, clean/reshape
import numpy as np     # Vectorized math, NaN handling, arrays, random utilities

# 2) Static charts for exploratory analysis
import matplotlib.pyplot as plt  # Base plotting backend; seaborn builds on this
import seaborn as sns            # High-level statistical plots (heatmap, pairplot, distplot, etc.)

# 3) Interactive charts for analysis/reports
import plotly.express as px       # Declarative API for quick interactive figs
import plotly.subplots as sp      # Helpers to arrange multiple Plotly charts
import plotly.graph_objects as go # Low-level, fully controllable Plotly figures and traces


In [39]:
# ============================================================
# step_02 LOAD RAW DATASET (CSV → DataFrame)
# ============================================================
# - Convert the uploaded CSV into a pandas DataFrame for EDA/modeling.

df = pd.read_csv('../data/raw/insurance_claims.csv')

In [40]:
# ============================================================
# step_03 VERIFY DATA INTEGRITY (SHAPE & MEMORY)
# ============================================================
# Why:
# - Confirm the dataset actually loaded (non-zero rows/columns).
# - Measure RAM footprint to judge if in-memory EDA/modeling is feasible.
# - Early tripwire for weirdness (e.g., shape (0, 0), gigantic memory use).

# 1) Report basic shape (#rows, #cols)
print(f"Dataset shape: {df.shape}")  # e.g., (1000, 40)

# 2) Report deep memory usage in MB (includes object/string columns)
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")



# Observation:

# The dataset size is manageable for local analysis without distributed computing needs.
# Dataset contains 1000 rows × 40 columns.
# Each row likely represents an individual insurance claim record.


Dataset shape: (1000, 40)
Memory usage: 1.28 MB


In [41]:
# ============================================================
# step_04 EXPLORE STRUCTURE & METADATA
# ============================================================
# Why:
# - Understand schema (column names, dtypes, non-null counts) to plan preprocessing.
# - Glance at distributions across numeric and categorical columns.
# - Inspect missingness patterns to prioritize cleaning steps.
# - Peek at a few rows to validate parsing (dates, IDs, categorical spellings).

# 1) Schema summary: dtypes and non-null counts
df.info()

# 2) Descriptive statistics across ALL dtypes (transpose for readability)
df.describe(include='all').T.head(15)

# 3) Missingness overview: top-10 columns with most NaNs
df.isna().sum().sort_values(ascending=False).head(10)

# 4) Quick row sample to sanity-check parsing
print(df.head(3))



# Observation:

# Initial scan confirms presence of both numerical and categorical variables with mixed data types.
# Some columns (e.g. _c39) appear empty or structural artifacts from CSV export.


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   months_as_customer           1000 non-null   int64  
 1   age                          1000 non-null   int64  
 2   policy_number                1000 non-null   int64  
 3   policy_bind_date             1000 non-null   object 
 4   policy_state                 1000 non-null   object 
 5   policy_csl                   1000 non-null   object 
 6   policy_deductable            1000 non-null   int64  
 7   policy_annual_premium        1000 non-null   float64
 8   umbrella_limit               1000 non-null   int64  
 9   insured_zip                  1000 non-null   int64  
 10  insured_sex                  1000 non-null   object 
 11  insured_education_level      1000 non-null   object 
 12  insured_occupation           1000 non-null   object 
 13  insured_hobbies    

In [42]:
# ============================================================
# step_05 AUDIT CATEGORICAL VALUES (UNIQUES SCAN)
# ============================================================
# Why:
# - Surface typos/inconsistent casing (e.g., "Yes"/"YES"/"yes") and stray tokens.
# - Spot sentinel strings ("?","N/A","-","Unknown") masquerading as categories.
# - Gauge category cardinality to plan encoding (OHE vs. hashing/target encoding).
# - Identify rare buckets to consider grouping or "Other" binning.

# 1) Iterate over all object-typed (categorical) columns
for col in df.select_dtypes(include='object').columns:
    # 2) Get distinct non-null tokens
    uniques = df[col].dropna().unique()
    # 3) Print a readable header and counts for manual inspection
    print(f"\n--- {col} ---")
    print(uniques)
    print(f"Unique count: {len(uniques)}")



--- policy_bind_date ---
['2014-10-17' '2006-06-27' '2000-09-06' '1990-05-25' '2014-06-06'
 '2006-10-12' '2000-06-04' '1990-02-03' '1997-02-05' '2011-07-25'
 '2002-05-26' '1999-05-29' '1997-11-20' '2012-10-26' '1998-12-28'
 '1992-10-19' '2005-06-08' '2004-11-15' '2014-12-28' '1992-08-02'
 '2002-06-25' '2005-11-27' '1994-05-27' '1991-02-08' '1996-02-02'
 '2013-12-05' '1990-09-20' '2002-07-18' '1990-02-08' '2014-03-04'
 '2000-02-18' '2008-06-19' '2003-08-01' '1992-04-04' '1991-01-13'
 '2010-08-08' '2003-03-09' '1993-02-03' '2002-11-25' '1997-07-27'
 '1995-05-08' '2012-08-30' '2006-04-30' '2003-04-13' '2007-12-05'
 '2006-08-21' '1990-01-08' '1990-03-18' '2008-01-07' '1997-03-10'
 '2004-01-10' '1994-08-20' '2002-06-02' '1994-04-28' '2014-08-17'
 '2007-08-11' '2008-11-20' '1995-10-18' '1993-05-19' '2005-02-26'
 '1990-02-14' '1993-09-30' '2014-06-10' '2008-10-28' '2007-04-24'
 '1993-12-13' '2011-08-17' '1990-08-20' '1994-02-11' '1994-02-22'
 '2011-08-15' '1998-11-11' '1995-12-04' '2002-12-2

In [43]:
# ============================================================
# step_06 CLEAN TEXT COLUMNS (PLACEHOLDERS & STRINGS)
# ============================================================
# Why:
# - Find “fake” missing values like "?", NA, N/A, NULL and turn them into real missing (NaN).
# - Trim spaces so we don’t create duplicates like " Yes" vs "Yes".
# - Spot empty strings and columns that mix types (text + numbers).
# - Warn about very rare categories (fewer than 5 rows) so we can group them later.

# 0) # 1) Create a working copy
df_model = df.copy()

# 1) Count "?" anywhere and by column
print("Total '?' found:", (df_model == "?").sum().sum())
print((df_model == "?").sum().sort_values(ascending=False))


# 2) Look for common placeholder strings (case/space-insensitive) in text columns
placeholders = ["NA","N/A","n/a","na","None","NULL","null","missing","Miss","Nan","nan"]
for p in placeholders:
    cols = df_model.select_dtypes('object').apply(lambda col: col.str.strip().str.upper() == p.upper()).sum()
    cols = cols[cols > 0]
    if len(cols) > 0:
        print(f"Found '{p}' in:", cols)

# 3) Trim spaces in all text columns
for col in df_model.select_dtypes('object'):
    df_model[col] = df_model[col].str.strip()

# 4) Count empty strings ("") after trimming
empties = (df_model.select_dtypes('object') == "").sum()
print("Empty strings per col:\n", empties[empties > 0])


# 4b) Count real NaN/None values in text columns
na_counts = df_model.select_dtypes('object').isna().sum()
na_counts = na_counts[na_counts > 0]

if len(na_counts) > 0:
    print("\nNaN/None values per column:")
    print(na_counts)
else:
    print("\nNo native NaN/None values found in text columns.")


# 5) Find columns that mix Python types (e.g., strings + floats)
for col in df_model.columns:
    if df_model[col].apply(type).nunique() > 1:
        print(f"⚠️ Mixed types in column '{col}'")

# 6) Flag rare categories (<5 rows) in text columns
for col in df_model.select_dtypes('object'):
    vc = df_model[col].value_counts()
    rares = vc[vc < 5]
    if len(rares) > 0:
        print(f"⚠️ Rare categories in {col}: {len(rares)}")

# ------------------------------------------------------------
# Observations from this run (plain language)
# ------------------------------------------------------------
# - We found 881 placeholder "?" values concentrated in:
#     • property_damage (360)
#     • police_report_available (343)
#     • collision_type (178)
#   → These are not real categories and should be treated as missing.
#
# - No empty strings detected → minimal whitespace noise.
#
# - authorities_contacted contains mixed Python types and 91 true NaN values
#   → standardize to string dtype and flag missingness.
#
# - policy_bind_date (951 uniques) and incident_location (1000 uniques)
#   show extremely high cardinality → behave like IDs, poor for one-hot encoding.
#   Better: drop entirely or extract components (e.g., year/month, region).
#
# - Missingness is informative (e.g., no police report) → create *_missing
#   indicator columns rather than blindly imputing.



Total '?' found: 881
property_damage                360
police_report_available        343
collision_type                 178
months_as_customer               0
policy_state                     0
age                              0
policy_number                    0
policy_bind_date                 0
policy_annual_premium            0
policy_deductable                0
policy_csl                       0
umbrella_limit                   0
insured_occupation               0
insured_hobbies                  0
insured_sex                      0
insured_zip                      0
capital-gains                    0
capital-loss                     0
incident_date                    0
incident_type                    0
incident_severity                0
authorities_contacted            0
insured_relationship             0
insured_education_level          0
incident_city                    0
incident_state                   0
incident_hour_of_the_day         0
incident_location                0

In [44]:
# ============================================================
# step_07 HANDLE MISSING VALUES IN CATEGORICAL VARIABLES
# ============================================================
# Why:
# - In fraud data, “missing” can be meaningful (e.g., no police report).
# - Here, missingness appears as both real NaN AND '?' placeholder strings.
# - We add *_missing flags so the model can learn from absence itself.
# - Then we fill missing values with the literal 'Missing'.



# 1) Replace placeholder '?' with real NaN FIRST
df_model = df_model.replace("?", np.nan)

# 2) Categorical columns where missing (NaN/?) occurs
missing_cols = [
    "collision_type",
    "property_damage",
    "police_report_available",
    "authorities_contacted"   # <-- real NaN found!
]

# 3) Add missing flags + fill
for col in missing_cols:

    # flag whether original value was missing
    df_model[col + "_missing"] = df_model[col].isna().astype(int)

    # replace NaN with "Missing" category
    df_model[col] = df_model[col].astype("string").fillna("Missing")



# ============================================================
# QUICK SANITY CHECK: MISSING FLAGS VS ORIGINALS
# ============================================================
# Why:
# - Visually confirm our *_missing flags line up with the source columns.
# - Expectation:
#   * If the original value was NaN (now "Missing"), the *_missing flag should be 1.
#   * If the original had a real value, the *_missing flag should be 0.
# - A small random sample is enough to spot obvious mismatches.

df_model[missing_cols + [col + "_missing" for col in missing_cols]].sample(8)  # sanity check

,collision_type,property_damage,police_report_available,authorities_contacted,collision_type_missing,property_damage_missing,police_report_available_missing,authorities_contacted_missing
819,Side Collision,YES,Missing,Other,0,0,1,0
967,Rear Collision,YES,NO,Police,0,0,0,0
809,Rear Collision,Missing,NO,Ambulance,0,1,0,0
304,Rear Collision,YES,NO,Other,0,0,0,0
736,Rear Collision,NO,Missing,Fire,0,0,1,0
54,Missing,NO,YES,Police,1,0,0,0
204,Rear Collision,NO,Missing,Ambulance,0,0,1,0
563,Rear Collision,YES,Missing,Police,0,0,1,0


In [45]:
# ============================================================
# step_08 BASELINE EXPERIMENT: CLAIM AMOUNTS — WHAT WORKS BEST?
# ============================================================
# Why:
# - Test simple ways to feed claim amounts to a model and see which works best:
#   A) the three parts (injury, property, vehicle)
#   B) the total only
#   C) parts + total (likely says the same thing twice)
#   D) total + shares (ratios of parts to total)
# - Shares put everything on the same scale, so huge claims don’t drown small ones.
# - Stratified K-fold keeps the fraud rate similar in every split.
# - We report:
#   PR-AUC  → handles imbalanced data well (higher is better)
#   ROC-AUC → ranking skill (higher is better)
#   Brier   → probability accuracy (lower is better)

# 1) Imports: CV splits, scaling, logistic regression, and scoring
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# 2) Target: tidy up 'fraud_reported' and map 'Y'/'N' → 1/0
y = df_model["fraud_reported"].astype(str).str.strip().str.upper().map({"Y": 1, "N": 0}).astype(int)

# 3) Build share features (parts divided by total)
#    Avoid divide-by-zero: replace 0 totals with NaN, then fill shares with 0.
tot = df_model["total_claim_amount"].replace(0, np.nan)
df_model["injury_share"]   = (df_model["injury_claim"]   / tot).fillna(0.0)
df_model["property_share"] = (df_model["property_claim"] / tot).fillna(0.0)
df_model["vehicle_share"]  = (df_model["vehicle_claim"]  / tot).fillna(0.0)  # implied by the other two; we won’t use it below

# 4) Feature sets to compare
feat_sets = {
    "A_components": ["injury_claim","property_claim","vehicle_claim"],
    "B_total": ["total_claim_amount"],
    "C_both": ["injury_claim","property_claim","vehicle_claim","total_claim_amount"],  # likely redundant
    "D_total_plus_shares": ["total_claim_amount","injury_share","property_share"],     # compact and scaled
}

# 5) Cross-validation helper: fit on folds, collect out-of-fold probabilities, score
def cv_eval(X, y, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, solver="lbfgs")),
    ])
    y_true_all, y_prob_all = [], []
    for tr, va in skf.split(X, y):
        pipe.fit(X.iloc[tr], y.iloc[tr])
        proba = pipe.predict_proba(X.iloc[va])[:, 1]
        y_true_all.append(y.iloc[va].values)
        y_prob_all.append(proba)
    y_true = np.concatenate(y_true_all)
    y_prob = np.concatenate(y_prob_all)
    return {
        "PR_AUC":  average_precision_score(y_true, y_prob),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "Brier":   brier_score_loss(y_true, y_prob),
    }

# 6) Run the comparison and print rounded scores
for name, cols in feat_sets.items():
    res = cv_eval(df_model[cols], y)
    print(name, {k: round(v, 3) for k, v in res.items()})




# ------------------------------------------------------------
# Interpretation (plain):
# D (total + shares) usually wins: it keeps scale info (total) and adds balanced ratios (shares).
# C (parts + total) often repeats the same signal and can be noisier.
# ------------------------------------------------------------


A_components {'PR_AUC': np.float64(0.298), 'ROC_AUC': np.float64(0.591), 'Brier': np.float64(0.182)}
B_total {'PR_AUC': np.float64(0.29), 'ROC_AUC': np.float64(0.593), 'Brier': np.float64(0.182)}
C_both {'PR_AUC': np.float64(0.298), 'ROC_AUC': np.float64(0.591), 'Brier': np.float64(0.182)}
D_total_plus_shares {'PR_AUC': np.float64(0.3), 'ROC_AUC': np.float64(0.593), 'Brier': np.float64(0.182)}


In [46]:
# ============================================================
# step_09 DROP COLUMNS AFTER EXPERIMENT
# ============================================================

df_model.drop(columns=["injury_share", "property_share", "vehicle_share"], inplace=True)


In [47]:
# ============================================================
# step_10 MAKE SHARE FEATURES (PARTS OF THE TOTAL)
# ============================================================
# Why:
# - We want to see how the total claim is split: how much was injury vs property.
# - Shares (ratios) make big and small claims comparable.
# - If total = 0, we avoid divide-by-zero and treat the share as 0.0.

# 1) Use total_claim_amount as the divider; turn 0 into NaN so we don’t divide by zero
tot = df_model["total_claim_amount"].replace(0, np.nan)

# 2) Shares = part / total. If total is missing or zero → fill share with 0.0
df_model["injury_share"]   = (df_model["injury_claim"]   / tot).fillna(0.0)  # fraction of the claim due to injuries
df_model["property_share"] = (df_model["property_claim"] / tot).fillna(0.0)  # fraction due to property damage

# Note:
# - We don’t store vehicle_share; it’s implied by the other two (the pieces add up to ~1 when total > 0).

In [48]:
# ============================================================
# step_11 MAKE TARGET NUMERIC (Y/N → 1/0)
# ============================================================
# Why:
# - Models expect numbers, not text labels.
# - We keep the original text column for reference, but add a numeric copy for modeling.
# - int8 saves memory while still holding 0/1 values.

# 1) Define mapping from text to numbers
y_map = {"Y": 1, "N": 0}

# 2) Create numeric target:
#    - clean spaces/case
#    - map Y/N to 1/0
#    - store as small integer
df_model["target"] = (
    df_model["fraud_reported"]
      .astype(str).str.strip().str.upper()  # normalize text
      .map(y_map)                           # Y/N → 1/0
      .astype("int8")                       # compact dtype
)


In [49]:
# ============================================================
# step_12 DROP RAW CLAIM COMPONENTS (NOW CAPTURED BY SHARES)
# ============================================================
# Why:
# - We already engineered injury_share/property_share from these.
# - Keeping the raw parts creates redundancy with total_claim_amount.
# - Dropping avoids multicollinearity and reduces feature noise.
# - total_claim_amount + shares preserve the full signal.

df_model.drop(columns=[
    "injury_claim",
    "property_claim",
    "vehicle_claim",
], inplace=True)


In [50]:
# ============================================================
# step_13 BASIC DOMAIN FIX: UMBRELLA LIMIT MUST BE NON-NEGATIVE
# ============================================================
# Why:
# - `umbrella_limit` is the max liability coverage on a policy; it can’t be negative.
# - A negative value is a data error (e.g., -1,000,000). We convert to absolute so values make sense.
# - Keep as numeric for modeling; it may carry risk signal.

# 1) Quick peek at distinct values (first 10) to spot bad signs like negatives
print("Unique umbrella limits:", df_model['umbrella_limit'].unique()[:10])

# 2) Fix: force non-negative values by taking absolute value
if "umbrella_limit" in df_model.columns:
    df_model["umbrella_limit"] = df_model["umbrella_limit"].abs()

# ------------------------------------------------------------
# Observation (from this dataset):
# - A single negative value (-1,000,000) was present and is now fixed.
# - Post-fix, values range approximately 0 to 10,000,000 (plausible coverage tiers).
# - We keep `umbrella_limit` as a numeric feature.
# ------------------------------------------------------------


Unique umbrella limits: [       0  5000000  6000000  4000000  3000000  8000000  7000000  9000000
 10000000 -1000000]


In [51]:
# ============================================================
# step_14 MAKE DATE FEATURES (KEEP/DROP RAW LATER)
# ============================================================
# Why:
# - Models don’t understand raw date strings; they do understand numbers.
# - Days between policy start and incident is a clear, useful signal.
# - Year/month capture seasonality without creating tons of dummy columns.
# - Day-of-week can catch weekly patterns (e.g., weekend spikes).
# - Bad data can produce negative gaps; we clip them to 0.

# 1) Parse text dates → real datetime (invalid parse → NaT)
for c in ["policy_bind_date", "incident_date"]:
    if c in df_model.columns:
        df_model[c] = pd.to_datetime(df_model[c], errors="coerce")

# 2) If both dates exist, build numeric + calendar features
if {"policy_bind_date","incident_date"}.issubset(df_model.columns):
    # 2a) Days from bind to incident; negatives → 0; missing → 0; store compact int
    d = (df_model["incident_date"] - df_model["policy_bind_date"]).dt.days
    df_model["days_since_bind"] = d.clip(lower=0).fillna(0).astype("int32")

    # 2b) Simple calendar parts (nullable ints so NaT stays as <NA>)
    df_model["incident_year"]  = df_model["incident_date"].dt.year.astype("Int64")
    df_model["incident_month"] = df_model["incident_date"].dt.month.astype("Int64")

    # 2c) Day-of-week features
    #     - incident_dow: 0=Mon … 6=Sun (nullable int)
    #     - incident_weekend: 1 if Sat/Sun else 0 (tiny int)
    df_model["incident_dow"]       = df_model["incident_date"].dt.dayofweek.astype("Int64")
    df_model["incident_weekend"]   = df_model["incident_dow"].isin([5, 6]).astype("Int8")

# Note: We’re not dropping the raw date columns in this cell.
#       Do that later in the modeling pipeline once all needed features are confirmed.


In [52]:
# ============================================================
# step_14b EXTRA DATE FEATURES FROM POLICY_BIND_DATE (WITH CHECKS)
# ============================================================
# Why:
# - Binding seasonality (month-of-year) often correlates with fraud:
#     renewal cycles, promo periods, policy shopping bursts.
# - Binding *year* can show vintage effects (older policies differ).
# - Binding *quarter* is a simpler seasonal bucket (Q1–Q4).
# - Combined with days_since_bind, this characterizes tenure risk.
# - After extracting these, we check whether any actually correlate
#   with observed fraud rates (target mean).

from IPython.display import display

if "policy_bind_date" in df_model.columns:

    # ------------------------------------------------------------
    # 1) Extract calendar parts (nullable integers preserved)
    # ------------------------------------------------------------
    df_model["policy_bind_year"]    = df_model["policy_bind_date"].dt.year.astype("Int64")
    df_model["policy_bind_month"]   = df_model["policy_bind_date"].dt.month.astype("Int64")
    df_model["policy_bind_quarter"] = df_model["policy_bind_date"].dt.quarter.astype("Int8")

    # ------------------------------------------------------------
    # 2) Quick visual sanity sample
    # ------------------------------------------------------------
    display(
        df_model[["policy_bind_year","policy_bind_month","policy_bind_quarter"]]
        .sample(8, random_state=42)
    )

    # ------------------------------------------------------------
    # 3) Fraud-rate diagnostics:
    #    Do year/month/quarter show useful separation?
    # ------------------------------------------------------------
    print("\nFraud rate by BIND MONTH:")
    display(
        df_model.groupby("policy_bind_month")["target"]
        .mean()
        .reset_index()
        .sort_values("target", ascending=False)
    )

    print("\nFraud rate by BIND QUARTER:")
    display(
        df_model.groupby("policy_bind_quarter")["target"]
        .mean()
        .reset_index()
        .sort_values("target", ascending=False)
    )

    print("\nFraud rate by BIND YEAR:")
    display(
        df_model.groupby("policy_bind_year")["target"]
        .mean()
        .reset_index()
        .sort_values("target", ascending=False)
    )

# ------------------------------------------------------------
# Interpretation (plain):
# - These tables reveal whether seasonality or vintage effects exist.
# - If patterns appear flat → features still harmless / cheap to keep.
# - If spikes appear in specific months/quarters → good predictive signal.
# ------------------------------------------------------------


,policy_bind_year,policy_bind_month,policy_bind_quarter
521,2014,7,3
737,2009,2,1
740,1990,7,3
660,1992,12,4
411,2003,4,2
678,2000,3,1
626,1997,3,1
513,1994,9,3



Fraud rate by BIND MONTH:


,policy_bind_month,target
4,5,0.318841
9,10,0.307692
1,2,0.300000
6,7,0.296703
3,4,0.289157
10,11,0.270588
5,6,0.262500
2,3,0.215190
0,1,0.204819
8,9,0.202703



Fraud rate by BIND QUARTER:


,policy_bind_quarter,target
1,2,0.288793
0,1,0.242063
3,4,0.236434
2,3,0.224806



Fraud rate by BIND YEAR:


,policy_bind_year,target
22,2012,0.388889
8,1998,0.333333
13,2003,0.324324
4,1994,0.317073
18,2008,0.303030
12,2002,0.300000
9,1999,0.275000
7,1997,0.272727
5,1995,0.256410
6,1996,0.255814


In [53]:
# ============================================================
# step_15 HOLIDAY EFFECTS (US, Q1 2015)
# ============================================================
# Why:
# - Holidays change driving patterns, alcohol use, travel volume,
#   and property theft risk. Insurance claim frequency changes
#   predictably around certain dates.
# - We map holiday dates in 2015 and create:
#     * is_holiday        → exact calendar match
#     * holiday_name      → categorical bucket (low cardinality)
#     * holiday_window_2d → ±2-day spillover window flag

from datetime import timedelta

# ------------------------------------------------------------
# 1) Define relevant 2015 US holiday dates
# ------------------------------------------------------------
holidays_2015 = {
    pd.Timestamp("2015-01-01"): "NewYears",
    pd.Timestamp("2015-01-19"): "MLK",
    pd.Timestamp("2015-02-01"): "SuperBowl",
    pd.Timestamp("2015-02-14"): "Valentines",   # optional
    pd.Timestamp("2015-02-16"): "Presidents",
}

# ------------------------------------------------------------
# 2) Exact-day holiday flag
# ------------------------------------------------------------
df_model["is_holiday"] = df_model["incident_date"].isin(holidays_2015.keys()).astype("int8")

# ------------------------------------------------------------
# 3) Holiday name category (non-holidays = "None")
# ------------------------------------------------------------
df_model["holiday_name"] = df_model["incident_date"].map(holidays_2015).fillna("None").astype("string")

# ------------------------------------------------------------
# 4) ±2-day spillover window (travel hangover, delays, alcohol)
# ------------------------------------------------------------
# Build a window set of all dates (minus/plus 2 days)
window = set()
for h in holidays_2015:
    window.update([h + timedelta(days=i) for i in range(-2, 3)])

df_model["holiday_window_2d"] = df_model["incident_date"].isin(window).astype("int8")

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------
print("\nCounts: Exact holiday hits")
print(df_model["is_holiday"].value_counts())

print("\nFraud rate by exact holiday:")
print(df_model.groupby("holiday_name")["target"].mean().round(3))

print("\nFraud rate by ±2-day window:")
print(df_model.groupby("holiday_window_2d")["target"].mean().round(3))



Counts: Exact holiday hits
is_holiday
0    906
1     94
Name: count, dtype: int64

Fraud rate by exact holiday:
holiday_name
MLK           0.391
NewYears      0.263
None          0.237
Presidents    0.250
SuperBowl     0.333
Valentines    0.444
Name: target, dtype: float64

Fraud rate by ±2-day window:
holiday_window_2d
0    0.241
1    0.258
Name: target, dtype: float64


In [54]:
# ============================================================
# step_16 DROP CONSTANT FEATURE INCIDENT_YEAR we have enginnered before (ZERO VARIANCE)
# ============================================================
# Why:
# - A column that has the same value for every row cannot help the model.
# - It only adds noise/complexity and can slightly hurt regularized models.

# Observation (from the stats above):
# - incident_year = 2015 for all 1,000 rows (std = 0.0) → no variation.


# Drop the constant column
if "incident_year" in df_model.columns:
    df_model.drop(columns=["incident_year"], inplace=True)


In [55]:
# ============================================================
# step_17 TIME-OF-DAY FEATURES (CYCLIC + SIMPLE BINS)
# ============================================================
# Why:
# - Hours wrap around (23 → 0). A plain number makes 23 and 0 look far apart.
# - Sin/Cos keeps the circle shape, so nearby hours stay “close”.
# - Simple buckets (Night/Morning/Afternoon/Evening) are easy to read and explain.

if "incident_hour_of_the_day" in df_model.columns:

    # 1) Make sure the hour is a number and within 0–23 (bad inputs get clipped)
    h = pd.to_numeric(df_model["incident_hour_of_the_day"], errors="coerce").clip(0, 23)

    # 2) Map the 24-hour clock onto a circle (in radians)
    angle = 2 * np.pi * (h / 24.0)

    # 3) Cyclic features: keep the wrap-around continuity
    df_model["hour_sin"] = np.sin(angle)
    df_model["hour_cos"] = np.cos(angle)

    # 4) Human-friendly time-of-day buckets for interpretation and tree splits
    bins_4   = [-0.1, 5, 11, 17, 23]
    labels_4 = ["Night (0-5)", "Morning (6-11)", "Afternoon (12-17)", "Evening (18-23)"]

    df_model["hour_bin_4"] = pd.cut(
        h,
        bins=bins_4,
        labels=labels_4,
        right=True,
        include_lowest=True
    )


In [56]:
# ============================================================
# step_18 SANITY CHECK: TIME-OF-DAY FEATURES (SHOW TABLES)
# ============================================================
# Why:
# - Double-check that hour → sin/cos/bins were created correctly.
# - Always DISPLAY each table so nothing is hidden by notebook output rules.
# - See counts per bucket and average fraud rate per bucket.

from IPython.display import display  # ensures all tables render

if {"incident_hour_of_the_day","hour_sin","hour_cos","hour_bin_4"}.issubset(df_model.columns):

    # 1) Quick sample of the raw hour + derived features
    display(df_model[["incident_hour_of_the_day","hour_sin","hour_cos","hour_bin_4"]].sample(10))

    # 2) Counts per time-of-day bucket (include NaN if any)
    vc = df_model["hour_bin_4"].value_counts(dropna=False).rename("count").reset_index()
    vc = vc.rename(columns={"index": "hour_bin_4"})
    display(vc)

    # 3) Fraud rate by bucket (mean of target), include NaN bucket if present
    fr = (
        df_model.groupby("hour_bin_4", dropna=False)["target"]
        .mean()
        .rename("fraud_rate")
        .reset_index()
    )
    display(fr)


,incident_hour_of_the_day,hour_sin,hour_cos,hour_bin_4
354,1,0.258819,9.659258e-01,Night (0-5)
516,17,-0.965926,-2.588190e-01,Afternoon (12-17)
804,3,0.707107,7.071068e-01,Night (0-5)
758,16,-0.866025,-5.000000e-01,Afternoon (12-17)
757,22,-0.500000,8.660254e-01,Evening (18-23)
98,6,1.000000,6.123234e-17,Morning (6-11)
696,17,-0.965926,-2.588190e-01,Afternoon (12-17)
156,10,0.500000,-8.660254e-01,Morning (6-11)
273,14,-0.500000,-8.660254e-01,Afternoon (12-17)
683,21,-0.707107,7.071068e-01,Evening (18-23)


,hour_bin_4,count
0,Afternoon (12-17),271
1,Evening (18-23),246
2,Night (0-5),244
3,Morning (6-11),239


/tmp/ipython-input-1406696201.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_model.groupby("hour_bin_4", dropna=False)["target"]


,hour_bin_4,fraud_rate
0,Night (0-5),0.241803
1,Morning (6-11),0.246862
2,Afternoon (12-17),0.250923
3,Evening (18-23),0.247967


In [57]:
# ============================================================
# step_19 EXPLORE HIGH-CARDINALITY CATEGORICALS
# ============================================================
# Why:
# - Columns with too many unique values (almost one per row) behave like IDs.
# - One-hot encoding them explodes features without useful patterns.
# - We’ll look at the top-10 “most unique” text columns and also check locations vs city.

# 1) Count unique values per text (object) column, sort by most unique
cat_counts = df_model.select_dtypes('object').nunique().sort_values(ascending=False)

# Show: top-10 categorical columns by # of unique values
print("\nTop-10 categorical columns by number of unique values:")
print(cat_counts.head(10))

# 2) Check whether 'incident_location' is basically unique per row by comparing to city
print("\nLocations per city (distinct incident_location within each city):")
print(df_model.groupby('incident_city')['incident_location'].nunique().sort_values(ascending=False))

print("\nSanity check — counts:")
print("Distinct cities:", df_model['incident_city'].nunique())
print("Distinct locations:", df_model['incident_location'].nunique())


# ------------------------------------------------------------
# Observations (plain):
# - 'incident_location' has ~1000 unique values (one per record) → acts like an ID.
# - Each city contains ~120–150 distinct locations; each location appears once.
# Action:
# - Don’t one-hot 'incident_location'. Prefer higher-level fields (city/state) or drop it.
# ------------------------------------------------------------



Top-10 categorical columns by number of unique values:
incident_location          1000
auto_model                   39
insured_hobbies              20
insured_occupation           14
auto_make                    14
incident_city                 7
insured_education_level       7
incident_state                7
insured_relationship          6
incident_type                 4
dtype: int64

Locations per city (distinct incident_location within each city):
incident_city
Springfield    157
Arlington      152
Columbus       149
Northbend      145
Hillsdale      141
Riverwood      134
Northbrook     122
Name: incident_location, dtype: int64

Sanity check — counts:
Distinct cities: 7
Distinct locations: 1000


In [58]:
# ============================================================
# step_20 DERIVE VEHICLE AGE (YEAR → AGE IN YEARS)
# ============================================================
# Why:
# - Fraud and claim size often relate to how old the car is, not the exact year.
# - Age is easier for a model to learn than raw year numbers.
# - In this dataset, incidents are in 2015, so age = 2015 - auto_year.
# - Note: if auto_year is missing, vehicle_age will be NaN too (that’s expected).

# 1) Create the age feature from the car’s model year
df_model['vehicle_age'] = 2015 - df_model['auto_year']

# 2) Quick preview to check the new column looks reasonable
df_model[['umbrella_limit', 'vehicle_age']].head()

# Observation:
# - vehicle_age captures depreciation and may help explain claim amounts and fraud patterns.


,umbrella_limit,vehicle_age
0,0,11
1,5000000,8
2,5000000,8
3,6000000,1
4,6000000,6


In [59]:
# ============================================================
# step_21 DROP REDUNDANT RAW YEAR (KEEP VEHICLE_AGE)
# ============================================================
# Why:
# - `auto_year` and `vehicle_age` say the same thing in different ways.
# - Keeping both can confuse the model and adds no new signal.
# - We keep the clearer feature (`vehicle_age`) and drop the duplicate.

# 1) Remove the raw year now that we have vehicle_age
df_model.drop(columns=['auto_year'], inplace=True)


In [60]:
# ============================================================
# step_22 CHECK CARDINALITY RATIOS (UNIQUES ÷ ROWS)
# ============================================================
# Why:
# - If a text column has a very high uniqueness ratio (e.g., >30%), it behaves like an ID.
# - Such columns make the model memorize records instead of learning general patterns.
# - Ratios near 0% mean values repeat a lot → safer to keep and encode.

# 1) Compute uniqueness ratio for each text column and sort high → low
cardinality_ratios = (
    df_model.select_dtypes('object').nunique() / len(df_model)
).sort_values(ascending=False)

# 2) Display the ratios for review
cardinality_ratios

# ------------------------------------------------------------
# Observations (plain):
# - After dropping 'incident_location' (~100% unique) and 'insured_zip' (~99.5% unique),
#   the remaining text columns show low uniqueness (~0.2% to ~3.9%).
# - These columns repeat across many rows, so they represent real groups, not row IDs.
# Action:
# - No more high-cardinality drops needed. Keep them so the model can learn patterns
#   (e.g., occupation, vehicle type, incident severity) linked to fraud risk.
# ------------------------------------------------------------


,0
incident_location,1.000
auto_model,0.039
insured_hobbies,0.020
insured_occupation,0.014
auto_make,0.014
incident_city,0.007
insured_education_level,0.007
incident_state,0.007
insured_relationship,0.006
incident_type,0.004


In [61]:
# ============================================================
# step_23 EXCLUSIVE CATEGORY CHECK (HOBBIES)
# ============================================================
# Why:
# - If a hobby shows up ONLY in fraud (target=1) or ONLY in non-fraud (target=0),
#   the model can just memorize that hobby instead of learning real patterns.
# - That’s risky: it can inflate scores now and fail on new data.
# - We’ll flag such “exclusive” categories so we can review/merge them later.

# 1) For each hobby, collect the set of target values it appears with (e.g., {0,1}, {1}, or {0})
exclusive = (
    df_model.groupby('insured_hobbies')['target']
    .apply(lambda x: set(x))    # unique target values for this hobby
    .reset_index()              # back to a DataFrame: columns = [insured_hobbies, target_set]
)

# 2) Keep only hobbies whose target set has size 1 → appears with only 0 or only 1
exclusive_only = exclusive[exclusive['target'].apply(lambda s: len(s) == 1)]

# 3) Show the flagged hobbies (review for leakage or merge to "Other")
exclusive_only

# Observation:

# There are no exclusive hobbies


,insured_hobbies,target


In [62]:
# ================================================================
# step_24 HOBBY-LEVEL LEAKAGE / RARE CATEGORY CHECK
# ================================================================
# Why:
# - Rare categories (very few rows) are easy for a model to memorize → overfitting.
# - A hobby with an extreme fraud rate (e.g., 100%) on tiny counts is suspicious.
# - We’ll list, for each hobby: how many rows it has and its fraud rate.

# 1) Group by hobby and compute size + fraud rate
hobby_stats = (
    df_model.groupby('insured_hobbies')
    .agg(
        total=('target', 'size'),        # how many rows for this hobby?
        fraud_rate=('target', 'mean')    # share of fraud for this hobby
    )
    .sort_values('fraud_rate', ascending=False)
)

# 2) Show the summary table
hobby_stats

# ------------------------------------------------------------
# Observation (plain):
# - No exclusive hobbies: every hobby appears with both classes (0 and 1) at least once.
# - Use this table to spot very small totals (e.g., <5) and extreme fraud rates
#   that might need grouping into "Other" or smoothing.
# ------------------------------------------------------------


,total,fraud_rate
insured_hobbies,,
chess,46,0.826087
cross-fit,35,0.742857
yachting,53,0.301887
board-games,48,0.291667
polo,47,0.276596
reading,64,0.265625
base-jumping,49,0.265306
hiking,52,0.230769
paintball,57,0.228070


In [63]:
# ============================================================
# step_25 FLAG SUSPICIOUS HOBBIES (POSSIBLE LEAKAGE)
# ============================================================
# Why:
# - Very high fraud rates on tiny categories can be dataset artifacts the model memorizes.
# - We’ll flag hobbies that look risky so we can review/merge them later
#   instead of letting the model overfit.

# 1) Hobbies with >75% fraud rate — likely artifacts, worth a closer look
leaky = hobby_stats[hobby_stats['fraud_rate'] > 0.75]
leaky  # inspect

# 2) Create a simple warning flag:
#    - total < 5  → too few rows (easy to memorize)
#    - fraud_rate > 0.70 → unusually high fraud share
hobby_stats['likely_leak'] = (
    (hobby_stats['total'] < 5) |
    (hobby_stats['fraud_rate'] > 0.7)
)

# 3) Show the table with the leak flag for manual review
hobby_stats


,total,fraud_rate,likely_leak
insured_hobbies,,,
chess,46,0.826087,True
cross-fit,35,0.742857,True
yachting,53,0.301887,False
board-games,48,0.291667,False
polo,47,0.276596,False
reading,64,0.265625,False
base-jumping,49,0.265306,False
hiking,52,0.230769,False
paintball,57,0.228070,False


In [64]:
# ============================================================
# step_26 MERGE RARE / RISKY HOBBIES INTO "OTHER"
# ============================================================
# Why:
# - Some hobbies are too rare (<5 rows) or have very high fraud rates (>70%).
# - Models can memorize these tiny categories → overfitting and fake “skill”.
# - Merging them into a single "Other" keeps the signal but removes the shortcut.
#
# Note:
# - This uses the earlier flag in hobby_stats['likely_leak'] from step_30.
# - If your modeling uses df_model (not df), mirror the replacement there too.

# 1) Get the list of risky hobbies we flagged earlier
leaky_hobbies = hobby_stats.loc[hobby_stats['likely_leak']].index.tolist()

# 2) Replace those hobbies with a safe combined bucket
df['insured_hobbies'] = df['insured_hobbies'].replace(leaky_hobbies, 'Other')

# 3) Quick check: see the new counts
print(df['insured_hobbies'].value_counts())


insured_hobbies
Other             81
reading           64
paintball         57
exercise          57
bungie-jumping    56
camping           55
golf              55
movies            55
kayaking          54
yachting          53
hiking            52
video-games       50
base-jumping      49
skydiving         49
board-games       48
polo              47
dancing           43
sleeping          41
basketball        34
Name: count, dtype: int64


In [65]:
# ============================================================
# step_27 PARSE POLICY_CSL TEXT INTO NUMERIC LIMITS
# ============================================================
# Why:
# - 'policy_csl' is text like "100/300" (per-person / per-incident liability cap).
# - Text hides the actual amounts; numbers are easier for models to learn.
# - Splitting gives two clear numeric features we can scale and interpret.

# 1) Safe parser: split "A/B" into two integers; if bad/missing → NaN, NaN
def parse_csl(value):
    try:
        a, b = value.split('/')
        return int(a), int(b)
    except:
        return np.nan, np.nan

# 2) Apply parsing row-by-row
parsed = df_model['policy_csl'].apply(parse_csl)

# 3) Create numeric columns from the parsed tuples
df_model['csl_per_person']   = parsed.apply(lambda x: x[0])  # e.g., 100
df_model['csl_per_incident'] = parsed.apply(lambda x: x[1])  # e.g., 300

# 4) Fill missing values with the column median (simple and robust)
df_model['csl_per_person'].fillna(df_model['csl_per_person'].median(), inplace=True)
df_model['csl_per_incident'].fillna(df_model['csl_per_incident'].median(), inplace=True)

# 5) Drop the original text column to avoid duplicates
df_model.drop(columns=['policy_csl'], inplace=True)

# 6) Quick preview of the new numeric features
df_model[['csl_per_person', 'csl_per_incident']].head()


/tmp/ipython-input-1565930519.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_model['csl_per_person'].fillna(df_model['csl_per_person'].median(), inplace=True)
/tmp/ipython-input-1565930519.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col

,csl_per_person,csl_per_incident
0,250,500
1,250,500
2,100,300
3,250,500
4,500,1000


In [66]:
# ============================================================
# step_28 PARSE INCIDENT_LOCATION INTO STREET INDEX + STREET TYPE
# ============================================================
# Why:
# - 'incident_location' looks like "4964 Elm Lane", "8014 Embarcadero Drive", etc.
# - The *number* part behaves like a near-unique identifier → useless for generalization.
# - The *suffix* (Lane, Drive, Ave, etc.) can hold mild signal:
#       different street categories → neighborhood type → fraud behavior differences.
# - We extract both and test uniqueness + fraud correlation.

# 1) Extract STREET TYPE (suffix) using regex (case-sensitive patterns)
#    Common street suffixes observed in the dataset:
street_suffixes = r'(Ave|Lane|Hwy|St|Drive|Ridge)$'
df_model["street_type"] = df_model["incident_location"].str.extract(street_suffixes)

# Fill anything that doesn't match into a neutral bucket
df_model["street_type"].fillna("Other", inplace=True)

# 2) Extract the STREET INDEX (leading numbers)
#    Anything before the first space that looks like digits:
df_model["street_index"] = df_model["incident_location"].str.extract(r'^(\d+)')
df_model["street_index"] = pd.to_numeric(df_model["street_index"], errors="coerce")

# 3) Quick uniqueness diagnostics
print("\nStreet TYPE counts (good — few categories):")
print(df_model["street_type"].value_counts())

print("\nStreet INDEX summary (bad — almost one per row):")
print(df_model["street_index"].describe())

print("Unique street indexes:", df_model["street_index"].nunique())

# 4) Fraud-rate by street type
print("\nFraud rate by STREET TYPE (possible weak signal):")
print(
    df_model.groupby("street_type")["target"]
    .mean()
    .sort_values(ascending=False)
)

# ------------------------------------------------------------
# Interpretation (plain):
# - street_type has ~6–7 categories → safe to keep and encode.
# - street_index has 939 unique values out of 1000 rows → behaves like a row-ID.
#   Keeping it would cause memorization and overfitting.
#
# Action:
# - Keep street_type (weak but harmless signal).
# - Drop street_index (high cardinality, no generalization).
# ------------------------------------------------------------

# 5) Final action: drop the near-unique index
df_model.drop(columns=["street_index"], inplace=True)
print("\nDropped column: street_index (quasi-identifier)")



Street TYPE counts (good — few categories):
street_type
Drive    173
Lane     171
Ridge    171
St       171
Ave      161
Hwy      153
Name: count, dtype: int64

Street INDEX summary (bad — almost one per row):
count    1000.000000
mean     5485.191000
std      2568.683425
min      1012.000000
25%      3280.250000
50%      5502.500000
75%      7657.750000
max      9988.000000
Name: street_index, dtype: float64
Unique street indexes: 939

Fraud rate by STREET TYPE (possible weak signal):
street_type
Lane     0.286550
Ridge    0.257310
Ave      0.254658
Drive    0.254335
St       0.228070
Hwy      0.196078
Name: target, dtype: float64

Dropped column: street_index (quasi-identifier)


/tmp/ipython-input-2125698646.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_model["street_type"].fillna("Other", inplace=True)


In [67]:
# ============================================================
# step_29 CITY vs STATE — DO WE NEED BOTH?
# ============================================================
# Why:
# - 'incident_city' and 'incident_state' are strongly coupled in this dataset:
#       • Every city appears in *all* 7 states (perfect symmetry)
#       • Every state contains all 7 cities
#   → They form a combinatorial grid, not real geography.
#
# - As a result:
#       • The model cannot generalize beyond this tiny synthetic pattern
#       • Including city adds noise, more dummy columns, and encourages overfitting
#
# - We will:
#       1) Verify uniqueness symmetry
#       2) Compare fraud signal of city vs state
#       3) Cross-validate predictive value
#       4) Decide which to keep
#

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import pandas as pd

print("Distinct values:")
display(df_model[['incident_state','incident_city']].nunique())

print("\nCities per state:")
display(df_model.groupby('incident_state')['incident_city'].nunique())

print("\nStates per city:")
display(df_model.groupby('incident_city')['incident_state'].nunique())

print("\nFraud rate by state:")
display(df_model.groupby('incident_state')['target'].mean().sort_values(ascending=False))

print("\nFraud rate by city:")
display(df_model.groupby('incident_city')['target'].mean().sort_values(ascending=False))


# ------------------------------------------------------------
# Quick model comparison: state-only vs state+city
# ------------------------------------------------------------
X_state = pd.get_dummies(df_model[['incident_state']])
X_both  = pd.get_dummies(df_model[['incident_state','incident_city']])
y       = df_model['target']

scores_state = cross_val_score(LogisticRegression(max_iter=2000), X_state, y,
                               cv=5, scoring='roc_auc')
scores_both  = cross_val_score(LogisticRegression(max_iter=2000), X_both,  y,
                               cv=5, scoring='roc_auc')

print("\nROC-AUC comparison (5-fold):")
print(f"State only: {scores_state.mean():.3f}")
print(f"State + City: {scores_both.mean():.3f}")

# ------------------------------------------------------------
# Interpretation (plain):
# - Fraud rates vary by state more than city.
# - Adding city *reduces* cross-validated ROC-AUC → mild overfitting.
# - Cities do not introduce new signal beyond state.
#
# Conclusion:
# - Keep 'incident_state' (compact + predictive).
# - Drop 'incident_city' to avoid extra dummies and noise.
# ------------------------------------------------------------

# FINAL ACTION
df_model.drop(columns=['incident_city'], inplace=True)
print("\nDropped column: incident_city (no added signal)")


Distinct values:


,0
incident_state,7
incident_city,7



Cities per state:


,incident_city
incident_state,
NC,7
NY,7
OH,7
PA,7
SC,7
VA,7
WV,7



States per city:


...


Fraud rate by state:


,target
incident_state,
OH,0.434783
NC,0.309091
SC,0.294355
PA,0.266667
VA,0.227273
NY,0.221374
WV,0.179724



Fraud rate by city:


,target
incident_city,
Arlington,0.289474
Columbus,0.261745
Hillsdale,0.248227
Springfield,0.242038
Northbend,0.234483
Riverwood,0.223881
Northbrook,0.221311



ROC-AUC comparison (5-fold):
State only: 0.562
State + City: 0.537

Dropped column: incident_city (no added signal)


In [68]:
# ============================================================
# step_30 VEHICLE FEATURES — AGE BUCKETS + MODEL-BASED TIER + BODY TYPE
# ============================================================
# Why:
# - Fraud pattern can depend on vehicle age (new vs old), perceived value (premium vs low),
#   and body type (SUV/pickup/sedan). These are interpretable, domain-plausible signals.
# - We avoid memorizing 39 distinct model names by compressing them into coarse buckets:
#       * premium  → expensive parts / higher payout incentives
#       * mid      → majority of fleet
#       * low      → ambiguous/cheap older models
# - Low cardinality keeps dummy columns small and reduces overfitting.

# ------------------------------------------------------------
# 1) VEHICLE AGE BUCKETS (simple 3-tier depreciation logic)
# ------------------------------------------------------------
# <= 3 years  → "new"
# <= 7 years  → "mid"
# else        → "old"

age_bins   = [0, 3, 7, 200]
age_labels = ["new", "mid", "old"]

df_model["vehicle_age_bucket"] = pd.cut(
    df_model["vehicle_age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

# ------------------------------------------------------------
# 2) VEHICLE TIER — MODEL-BASED (proxy for perceived luxury / payout risk)
# ------------------------------------------------------------
# Premium: high-value models with expensive parts & repairs
premium_models = {
    "M5", "X5", "X6", "3 Series",
    "E400", "C300", "ML350",
    "A3", "A5"
}

# Low-value: ambiguous / older Saab-style imports (92x/93/95 pattern)
low_models = {"92x", "93", "95"}

# Everything else → mid-market (largest bucket)

def map_vehicle_tier(model):
    if model in premium_models:
        return "premium"
    elif model in low_models:
        return "low"
    else:
        return "mid"

df_model["vehicle_tier"] = df_model["auto_model"].apply(map_vehicle_tier)

# ------------------------------------------------------------
# 3) BODY TYPE — simplified mapping from model names
# ------------------------------------------------------------
# Rough grouping based on common US insurance risk categories.
pickup = ["RAM", "F150", "Silverado"]
suv    = ["Tahoe", "Pathfinder", "Highlander", "X5", "CRV"]
sedan  = ["Camry", "Corolla", "Ultima", "Malibu", "A3", "A5", "C300"]

def body(model):
    if model in pickup: return "pickup"
    if model in suv:    return "suv"
    if model in sedan:  return "sedan"
    return "other"

df_model["body_type"] = df_model["auto_model"].map(body)

# ------------------------------------------------------------
# 4) QUICK DIAGNOSTICS — counts
# ------------------------------------------------------------
print("\nVehicle Age Buckets:")
print(df_model["vehicle_age_bucket"].value_counts())

print("\nVehicle Tier (model-based):")
print(df_model["vehicle_tier"].value_counts())

print("\nBody Type:")
print(df_model["body_type"].value_counts())

# ------------------------------------------------------------
# 5) FRAUD-RATE CHECKS (do these differentiate risk?)
# ------------------------------------------------------------
print("\nFraud by Age Bucket:")
print(df_model.groupby("vehicle_age_bucket")["target"].mean().round(3))

print("\nFraud by Vehicle Tier (model-based):")
print(df_model.groupby("vehicle_tier")["target"].mean().round(3))

print("\nFraud by Body Type:")
print(df_model.groupby("body_type")["target"].mean().round(3))

# ------------------------------------------------------------
# Interpretation (plain):
# - Each feature has low cardinality (≤ 6 categories) → safe to one-hot later.
# - "premium" shows mildly higher fraud → plausible signal.
# - pickup trucks show elevated fraud risk → relevant pattern.
# - Compressing 39 raw models prevents overfitting to ultra-rare variants.
# ------------------------------------------------------------



Vehicle Age Buckets:
vehicle_age_bucket
old    616
mid    198
new    186
Name: count, dtype: int64

Vehicle Tier (model-based):
vehicle_tier
mid        714
premium    206
low         80
Name: count, dtype: int64

Body Type:
body_type
other     600
sedan     188
suv       120
pickup     92
Name: count, dtype: int64

Fraud by Age Bucket:
vehicle_age_bucket
new    0.253
mid    0.227
old    0.252
Name: target, dtype: float64

Fraud by Vehicle Tier (model-based):
vehicle_tier
low        0.225
mid        0.232
premium    0.306
Name: target, dtype: float64

Fraud by Body Type:
body_type
other     0.240
pickup    0.359
sedan     0.218
suv       0.242
Name: target, dtype: float64


/tmp/ipython-input-2976789849.py:87: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_model.groupby("vehicle_age_bucket")["target"].mean().round(3))


In [69]:
# ============================================================
# step_31 DROP IRRELEVANT / REDUNDANT RAW COLUMNS
# ============================================================
# Why each column is removed:
# ------------------------------------------------------------
# "_c39"
#   • Completely empty column (0 non-null entries) → pure noise.
#
# "policy_number"
#   • Surrogate row identifier; model would memorize it → overfitting.
#
# "incident_location"
#   • ~1000 unique values (almost 1 per row). Behaves like an ID.
#   • We already extracted street_type; location adds no generalizable signal.
#
# "fraud_reported"
#   • We created the numeric 'target' column from this.
#   • Keeping the string duplicate would leak information into encoding.
#
# "incident_date"
#   • We extracted incident_month, incident_dow, incident_weekend, holiday flags.
#   • Raw date adds no new information, only parsing overhead.
#
# "policy_bind_date"
#   • We extracted policy_bind_year/month/quarter + days_since_bind.
#   • Raw date no longer needed → redundant.
#
# "incident_hour_of_the_day"
#   • Converted into cyclic features (hour_sin/hour_cos) and coarse bins.
#   • Raw integer is now redundant.
#
# "holiday_name"
#   • Very sparse, many tiny one-hot dummy columns if encoded.
#   • We already captured "is_holiday" and "holiday_window_2d"
#     (densely informative and cheaper to encode).
#
# "insured_zip"
#   • Is almost unique per row (acts like an ID, not a region) 995 out of 1000.
# ------------------------------------------------------------

cols_to_drop = [
    "_c39",
    "policy_number",
    "incident_location",
    "fraud_reported",
    "incident_date",
    "policy_bind_date",
    "incident_hour_of_the_day",
    "holiday_name",
    "insured_zip"
]

df_model.drop(columns=cols_to_drop, inplace=True)

print(f"Dropped {len(cols_to_drop)} columns. New shape: {df_model.shape}")


Dropped 9 columns. New shape: (1000, 52)


In [70]:
# ============================================================
# step_32 FINAL COUNT UNIQUE VALUES IN TEXT COLUMNS
# ============================================================
# Why:
# - Text columns with too many unique values behave like hidden IDs.
# - These can make models memorize records instead of learning real patterns.
# - We list how many distinct values each text column has, from most to least.

# 1) Show unique-counts per categorical (object) column, sorted high → low
df_model.select_dtypes('object').nunique().sort_values(ascending=False)



,0
auto_model,39
insured_hobbies,20
auto_make,14
insured_occupation,14
insured_education_level,7
incident_state,7
insured_relationship,6
street_type,6
incident_type,4
incident_severity,4


In [71]:
# ============================================================
# step_33 FINAL SANITY REVIEW OF MODEL DATAFRAME
# ============================================================
# Why:
# - Quick, human-readable audit before training.
# - Check size, memory, dtypes, missing values, target balance,
#   numeric ranges, categorical cardinality, and accidental leftovers.

from IPython.display import display

# 1) Shape & memory
print(f"Shape (rows, cols): {df_model.shape}")
print(f"Memory usage (MB): {df_model.memory_usage(deep=True).sum() / 1024**2:.2f}")

# 2) Schema (dtypes, non-null counts)
df_model.info()

# 3) Target balance
if "target" in df_model.columns:
    vc = df_model["target"].value_counts(dropna=False).rename("count")
    pct = (vc / len(df_model)).rename("ratio")
    display(
        pd.concat([vc, pct.round(3)], axis=1)
          .rename_axis("target")
          .reset_index()
    )

# 4) Missing values by column (top 20)
na_counts = df_model.isna().sum().sort_values(ascending=False)
print("\nTop missing-value columns:")
display(na_counts.head(20))

# 5) Numeric ranges (mean/std/min/max)
num_cols = df_model.select_dtypes(include=["number"]).columns
if len(num_cols) > 0:
    num_summary = df_model[num_cols].describe().T[["mean","std","min","max"]]
    print("\nNumeric summary (mean/std/min/max):")
    display(num_summary)

# 6) Categorical cardinality (top 10)
cat_cols = df_model.select_dtypes(include=["object","string"]).columns
if len(cat_cols) > 0:
    cat_card = df_model[cat_cols].nunique().sort_values(ascending=False)
    print("\nTop-10 categorical columns by # unique values:")
    display(cat_card.head(10))

# 7) Accidental leftovers check (should be absent)
should_be_dropped = ["incident_location","insured_zip","policy_csl","_c39","auto_year"]
still_present = [c for c in should_be_dropped if c in df_model.columns]
print("\nColumns that should have been dropped but are still present:", still_present or "None")

# 8) Duplicate rows (exact duplicates)
dupe_count = df_model.duplicated().sum()
print(f"\nDuplicate rows: {dupe_count}")

# 9) Small sample preview
print("\nSample rows:")
display(df_model.sample(5, random_state=42))


Shape (rows, cols): (1000, 52)
Memory usage (MB): 1.18
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 52 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   months_as_customer               1000 non-null   int64   
 1   age                              1000 non-null   int64   
 2   policy_state                     1000 non-null   object  
 3   policy_deductable                1000 non-null   int64   
 4   policy_annual_premium            1000 non-null   float64 
 5   umbrella_limit                   1000 non-null   int64   
 6   insured_sex                      1000 non-null   object  
 7   insured_education_level          1000 non-null   object  
 8   insured_occupation               1000 non-null   object  
 9   insured_hobbies                  1000 non-null   object  
 10  insured_relationship             1000 non-null   object  
 11  capital-gains  

,target,count,ratio
0,0,753,0.753
1,1,247,0.247



Top missing-value columns:


,0
months_as_customer,0
age,0
policy_state,0
policy_deductable,0
policy_annual_premium,0
umbrella_limit,0
insured_sex,0
insured_education_level,0
insured_occupation,0
insured_hobbies,0



Numeric summary (mean/std/min/max):


,mean,std,min,max
months_as_customer,203.954,115.113174,0.0,479.0
age,38.948,9.140287,19.0,64.0
policy_deductable,1136.0,611.864673,500.0,2000.0
policy_annual_premium,1256.40615,244.167395,433.33,2047.59
umbrella_limit,1103000.0,2296446.094439,0.0,10000000.0
capital-gains,25126.1,27872.187708,0.0,100500.0
capital-loss,-26793.7,28104.096686,-111100.0,0.0
number_of_vehicles_involved,1.839,1.01888,1.0,4.0
bodily_injuries,0.992,0.820127,0.0,2.0
witnesses,1.487,1.111335,0.0,3.0



Top-10 categorical columns by # unique values:


,0
auto_model,39
insured_hobbies,20
auto_make,14
insured_occupation,14
insured_education_level,7
incident_state,7
street_type,6
insured_relationship,6
authorities_contacted,5
incident_severity,4



Columns that should have been dropped but are still present: None

Duplicate rows: 0

Sample rows:


,months_as_customer,age,policy_state,policy_deductable,policy_annual_premium,umbrella_limit,insured_sex,insured_education_level,insured_occupation,insured_hobbies,...,hour_sin,hour_cos,hour_bin_4,vehicle_age,csl_per_person,csl_per_incident,street_type,vehicle_age_bucket,vehicle_tier,body_type
521,5,26,IL,2000,1137.02,0,FEMALE,PhD,farming-fishing,skydiving,...,-0.500000,8.660254e-01,Evening (18-23),12,250,500,Hwy,old,premium,sedan
737,160,33,IL,1000,1422.78,0,FEMALE,High School,exec-managerial,exercise,...,-0.965926,-2.588190e-01,Afternoon (12-17),9,500,1000,Lane,old,mid,suv
740,385,51,IN,1000,976.37,0,FEMALE,MD,craft-repair,reading,...,-0.500000,-8.660254e-01,Afternoon (12-17),8,250,500,Ave,old,mid,other
660,446,57,IN,2000,1373.21,0,MALE,College,adm-clerical,sleeping,...,0.500000,-8.660254e-01,Morning (6-11),3,100,300,Ridge,new,mid,other
411,84,29,OH,1000,1117.17,0,FEMALE,High School,machine-op-inspct,video-games,...,1.000000,6.123234e-17,Morning (6-11),10,250,500,Ave,old,premium,other
